# ASL Isolated Sign Recognition — EDA

Exploratory analysis of the Kaggle *asl-signs* dataset (Google - Isolated Sign Language
Recognition), used to pin down the modelling choices that follow:

1. Sample distribution across all 250 signs
2. Selection of the 30-sign subset used for the rest of the project
3. Sequence length distribution → the `max_len` used for padding
4. Skeleton visualisation, as a sanity check that the landmarks decode correctly
5. Participant counts → confirming a signer-disjoint split is viable

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Walk up from the working directory to find the repo root, so the notebook runs
# whether the kernel starts in notebooks/ or at the project root.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import make_splits, select_top_signs

DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
BLUE, GREEN, RED = "#4C72B0", "#55A868", "#C44E52"

train_df = pd.read_csv(DATA_DIR / "train.csv")
print(f"train.csv: {train_df.shape[0]} sequences, {train_df.shape[1]} columns")
train_df.head()

## 1. Samples per sign

How balanced is the label space across all 250 signs?

In [ ]:
sign_counts = train_df["sign"].value_counts()

print(f"{len(sign_counts)} unique signs")
print(f"samples per sign: min {sign_counts.min()}, max {sign_counts.max()}, mean {sign_counts.mean():.0f}")
print(f"imbalance ratio (max/min): {sign_counts.max() / sign_counts.min():.2f}x")

fig, ax = plt.subplots(figsize=(15, 4.5))
ax.bar(range(len(sign_counts)), sign_counts.to_numpy(), color=BLUE, width=1.0)
ax.set(
    xlabel="sign (sorted by sample count, descending)",
    ylabel="samples",
    title=f"Samples per sign — all {len(sign_counts)} signs",
)
ax.set_xlim(-1, len(sign_counts))
plt.tight_layout()
plt.show()

## 2. The 30-sign subset

Everything downstream uses only the 30 most frequent signs: it keeps training fast and the
class space manageable while still being a non-trivial classification problem.

`select_top_signs` lives in `src/dataset.py` so the notebook and the training pipeline derive
the identical vocabulary (ties broken by name, result sorted alphabetically) rather than
depending on a saved file that could drift.

In [ ]:
TOP_N = 30
top_signs = select_top_signs(train_df, num_signs=TOP_N)
subset_df = train_df[train_df["sign"].isin(top_signs)].reset_index(drop=True)

signs_path = PROCESSED_DIR / "top30_signs.json"
signs_path.write_text(json.dumps(top_signs, indent=2), encoding="utf-8")

print(f"{len(subset_df)} sequences ({100 * len(subset_df) / len(train_df):.1f}% of the full dataset)")
print(f"saved vocabulary -> {signs_path.relative_to(PROJECT_ROOT)}")
print(top_signs)

top_counts = subset_df["sign"].value_counts()

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top_counts.index[::-1], top_counts.to_numpy()[::-1], color=BLUE)
ax.set(xlabel="samples", title=f"Top {TOP_N} signs by sample count")
plt.tight_layout()
plt.show()

## 3. Sequence lengths

Frame counts drive the `max_len` used for padding/truncation. Reading every parquet file in the
subset takes a few minutes, so the result is cached to `data/processed/`.

In [ ]:
lengths_path = PROCESSED_DIR / "sequence_lengths.csv"

if lengths_path.exists():
    lengths_df = pd.read_csv(lengths_path)
    print(f"loaded cached lengths from {lengths_path.relative_to(PROJECT_ROOT)}")
else:
    records = []
    for i, row in enumerate(subset_df.itertuples(), start=1):
        frames = pd.read_parquet(DATA_DIR / row.path, columns=["frame"])
        records.append({"sequence_id": row.sequence_id, "num_frames": frames["frame"].nunique()})
        if i % 1000 == 0:
            print(f"  read {i}/{len(subset_df)} sequences")
    lengths_df = pd.DataFrame(records)
    lengths_df.to_csv(lengths_path, index=False)
    print(f"cached lengths -> {lengths_path.relative_to(PROJECT_ROOT)}")

lengths = lengths_df["num_frames"]
print()
print(f"min    {lengths.min()}")
print(f"max    {lengths.max()}")
print(f"mean   {lengths.mean():.1f}")
print(f"median {lengths.median():.0f}")
print(f"p95    {lengths.quantile(0.95):.0f}")
print(f"p99    {lengths.quantile(0.99):.0f}")

In [ ]:
MAX_LEN = 64  # DEFAULT_MAX_LEN in src/dataset.py — the coverage below is the justification

covered = 100 * (lengths <= MAX_LEN).mean()
real_fraction = 100 * np.minimum(lengths, MAX_LEN).mean() / MAX_LEN
print(f"sequences fully covered by max_len={MAX_LEN}: {covered:.1f}%")
print(f"share of each padded batch that is real data: {real_fraction:.1f}%")
print(f"the rest are uniformly subsampled to {MAX_LEN} frames, not truncated")

# The distribution is heavily right-skewed (median well below the mean), so a larger max_len
# buys little extra coverage while turning most of every batch into padding.
print()
print(f"{'max_len':>8} {'covered':>9} {'real data':>10} {'attn cost':>10}")
for candidate in [32, 64, 96, 128, 256]:
    cov = 100 * (lengths <= candidate).mean()
    real = 100 * np.minimum(lengths, candidate).mean() / candidate
    print(f"{candidate:>8} {cov:>8.1f}% {real:>9.1f}% {(candidate / MAX_LEN) ** 2:>9.2f}x")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, log in zip(axes, [False, True]):
    ax.hist(lengths, bins=60, color=BLUE)
    ax.axvline(MAX_LEN, color=RED, ls="--", label=f"max_len = {MAX_LEN}")
    ax.set(xlabel="frames per sequence", ylabel="sequences")
    if log:
        ax.set_yscale("log")
        ax.set_title("Same distribution, log scale (tail detail)")
    else:
        ax.set_title(f"Sequence lengths — top {TOP_N} signs")
    ax.legend()
plt.tight_layout()
plt.show()

## 4. Skeleton sanity check

Decode one sequence and draw the pose and hand landmarks over time. If the skeleton looks like a
person signing, the landmark layout (type / landmark_index → x, y) is being read correctly.

In [ ]:
# MediaPipe hand topology: five fingers off the wrist, plus the knuckle line across the palm.
HAND_EDGES = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
    (5, 9), (9, 13), (13, 17),
]
# Upper body only — legs are never in frame for these recordings.
POSE_EDGES = [(11, 12), (11, 13), (13, 15), (12, 14), (14, 16), (11, 23), (12, 24), (23, 24)]

example = subset_df.iloc[0]
seq = pd.read_parquet(DATA_DIR / example["path"])
frames = sorted(seq["frame"].unique())
print(f'sign: "{example["sign"]}"  |  sequence_id: {example["sequence_id"]}  |  {len(frames)} frames')

visible = seq[seq["type"].isin(["pose", "left_hand", "right_hand"])]
bounds = (
    np.nanmin(visible["x"]), np.nanmax(visible["x"]),
    np.nanmin(visible["y"]), np.nanmax(visible["y"]),
)


def frame_xy(seq_df, frame, landmark_type):
    part = seq_df[(seq_df["frame"] == frame) & (seq_df["type"] == landmark_type)]
    return part.sort_values("landmark_index")[["x", "y"]].to_numpy()


def draw_skeleton(ax, seq_df, frame, bounds):
    for landmark_type, edges, color in [
        ("pose", POSE_EDGES, BLUE),
        ("left_hand", HAND_EDGES, GREEN),
        ("right_hand", HAND_EDGES, RED),
    ]:
        xy = frame_xy(seq_df, frame, landmark_type)
        if len(xy) == 0 or np.isnan(xy).all():
            continue
        ax.scatter(xy[:, 0], xy[:, 1], s=7, color=color, zorder=3)
        for a, b in edges:
            if a < len(xy) and b < len(xy) and not np.isnan(xy[[a, b]]).any():
                ax.plot(xy[[a, b], 0], xy[[a, b], 1], color=color, lw=1.2)
    x_lo, x_hi, y_lo, y_hi = bounds
    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(y_hi, y_lo)  # landmark y grows downward, so invert to match the image
    ax.set_title(f"frame {frame}", fontsize=9)
    ax.set_aspect("equal")
    ax.axis("off")


sampled = [frames[i] for i in np.linspace(0, len(frames) - 1, 8).round().astype(int)]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, frame in zip(axes.ravel(), sampled):
    draw_skeleton(ax, seq, frame, bounds)
fig.suptitle(f'"{example["sign"]}" — pose (blue), left hand (green), right hand (red)')
plt.tight_layout()
plt.show()

In [ ]:
# Same sequence as an inline animation (jshtml needs no ffmpeg installed).
from IPython.display import HTML
from matplotlib import animation

fig, ax = plt.subplots(figsize=(4.5, 4.5))
anim = animation.FuncAnimation(
    fig,
    lambda frame: (ax.clear(), draw_skeleton(ax, seq, frame, bounds)),
    frames=frames,
    interval=100,
)
plt.close(fig)
HTML(anim.to_jshtml())

## 5. Participants

A random split would leak the same signer into train and test, so the score would partly measure
memorisation of individual signers. Splitting by `participant_id` instead measures generalisation
to *unseen* signers — which needs enough participants to give every split a few.

In [ ]:
participant_counts = subset_df["participant_id"].value_counts().sort_index()
coverage = subset_df.groupby("participant_id")["sign"].nunique()

print(f"{len(participant_counts)} distinct participants in the {TOP_N}-sign subset")
print(f"sequences per participant: min {participant_counts.min()}, max {participant_counts.max()}, mean {participant_counts.mean():.0f}")
print(f"signs covered per participant: min {coverage.min()}, max {coverage.max()} (of {TOP_N})")

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(participant_counts.index.astype(str), participant_counts.to_numpy(), color=BLUE)
ax.set(xlabel="participant_id", ylabel="sequences", title=f"Sequences per participant — top {TOP_N} signs")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Preview the signer-disjoint split used by the training pipeline.
splits = make_splits(train_df)

for name, split_df in splits.items():
    in_subset = split_df[split_df["sign"].isin(top_signs)]
    print(
        f"{name:>5}: {split_df['participant_id'].nunique():>2} participants, "
        f"{len(in_subset):>5} sequences, {in_subset['sign'].nunique()}/{TOP_N} signs"
    )

train_ids = set(splits["train"]["participant_id"])
print()
print(f"train/val participant overlap:  {len(train_ids & set(splits['val']['participant_id']))} (must be 0)")
print(f"train/test participant overlap: {len(train_ids & set(splits['test']['participant_id']))} (must be 0)")

## Takeaways

- **Class balance:** the 250 signs are close to uniform (1.39x between the most and least
  frequent), so the 30-sign subset needs no resampling or class weighting.
- **`max_len = 64`:** the length distribution is heavily right-skewed (median 24, mean 39,
  max 453). 64 covers 84% of sequences outright and keeps ~47% of each padded batch as real
  data, where 128 would leave ~71% padding for 10 points more coverage — and it makes the
  Transformer's quadratic attention 4x cheaper. Longer sequences are uniformly subsampled,
  so a long sign keeps its ending at coarser resolution rather than losing it.
- **Landmarks decode correctly**, and hands drop out of frame often enough that missing-value
  handling matters (`src/dataset.py` interpolates over time before falling back to a zero fill).
- **Splits are signer-disjoint**, so val/test measure generalisation to new signers.

Next: baseline model over these fixed-length sequences.